# Reproducing MISO: Multi-Implicit-Submaps
Run the cells below to set up the environment and run the SLAM demo.

In [4]:
import chi
from chi import lease, server
from fabric import Connection
import os
import time
import sys 

chi.use_site("CHI@TACC") 
chi.set("project_name", "YOUR-PROJECT-ID")  # Your Project ID

print("Reserving a Tesla P100 GPU node...")
reservations = []
lease_node_type = "gpu_p100"

try:
    lease.add_fip_reservation(reservations, count=1)
    lease.add_node_reservation(reservations, node_type=lease_node_type, count=1)
    start, end = lease.lease_duration(hours=2)

    # Create the lease
    my_lease = lease.create_lease(
        f"{os.getenv('USER')}-reproduce-miso",
        reservations,
        start_date=start,
        end_date=end
    )
except Exception as e:
    print("Failed to create lease:")
    print(e)
    sys.exit(1)

print("Lease created, go to the GUI portal to check if the lease is active...")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org
Reserving a Tesla P100 GPU node...
Lease created, go to the GUI portal to check if the lease is active...


# Create the server
Wait for the lease to be active in the GUI portal. 

We have trouble using lease.wait_for_active(lease_id) because of the code inside the library. (Note: do tell me if we do something wrong)

In [5]:
image = "CC-Ubuntu20.04-CUDA"
lease_id = my_lease["id"]
reservation_id = my_lease['reservations'][0]['id']

my_server = server.create_server(
   f"{os.getenv('USER')}-power-management",
   image_name=image,
   reservation_id=reservation_id
)

print("Waiting for server to start ...")
server.wait_for_active(my_server.id)
print("Done")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server to start ...
Done


In [6]:
import socket
from chi import ssh, network

# Get the floating IP from your lease
print(f"Looking for available Floating IP for lease {lease_id}...")

# Get all floating IPs allocated to your project
all_fips = network.list_floating_ips()

# Filter for IPs that are not currently attached to any server (port_id is None)
available_fips = [f for f in all_fips if not f.get('port_id')]

if available_fips:
    # Pick the first available one
    floating_ip = available_fips[0]['floating_ip_address']
    print(f"Found available Floating IP: {floating_ip}")
else:
    # Fallback: If you are re-running this and it's already attached, 
    # we might need to find the one attached to your specific server ID if it exists.
    raise RuntimeError(
        "No unassociated Floating IPs found! "
        "Check if the IP is already attached or if the lease failed."
    )

# Associate it with your server
server.associate_floating_ip(my_server.id, floating_ip_address=floating_ip)

print(f"Waiting for SSH connectivity on {floating_ip} ...")

# Connection Polling
timeout = 120 # 2 minutes
start_time = time.perf_counter()
while True:
    try:
        # We use a short timeout for the socket check itself
        with socket.create_connection((floating_ip, 22), timeout=5):
            print("SSH port is open and responding!")
            break
    except (OSError, ConnectionRefusedError):
        elapsed = time.perf_counter() - start_time
        if elapsed >= timeout:
            print(f"After {timeout} seconds, could not connect via SSH.")
            break
        print(f"Still waiting... ({int(elapsed)}s elapsed)")
        time.sleep(10)

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Looking for available Floating IP for lease 3b8c04bc-b73a-495b-a1ac-06ba702f6f82...
Found available Floating IP: 129.114.109.70
Waiting for SSH connectivity on 129.114.109.70 ...
Still waiting... (1s elapsed)
Still waiting... (11s elapsed)
Still waiting... (21s elapsed)
SSH port is open and responding!


In [9]:
from chi import ssh
import os

GITHUB_REPO = "https://github.com/muhnatha/reproduce_MISO.git"

print(f"Connecting to {floating_ip}...")

with ssh.Remote(floating_ip) as conn:
    # Force-Kill System Locks
    print("1. Terminating background update processes (fixing the lock error)...")
    conn.run("sudo systemctl stop unattended-upgrades || true")
    conn.run("sudo systemctl stop apt-daily.service || true")
    conn.run("sudo systemctl stop apt-daily-upgrade.service || true")
    conn.run("sudo fuser -k -9 /var/lib/dpkg/lock-frontend || true")
    conn.run("sudo fuser -k -9 /var/lib/apt/lists/lock || true")
    conn.run("sudo dpkg --configure -a || true")

    # Clone Repository
    print("Cloning repository...")
    conn.run(f"rm -rf reproduce_miso && git clone {GITHUB_REPO} reproduce_miso")

    # Install Core System Dependencies
    print("Installing pip and system utilities...")
    conn.run("sudo apt-get update && sudo apt-get install -y unzip build-essential python3-pip python3-dev", pty=True)
    
    print("Upgrading pip...")
    conn.run("python3 -m pip install --upgrade pip", pty=True)

    # Install Torch & CUDA-specific wheels
    print("Installing PyTorch, fvcore, iopath, and PyTorch3D...")
    conn.run("python3 -m pip install torch==1.13.1+cu116 torchvision==0.14.1+cu116 --extra-index-url https://download.pytorch.org/whl/cu116", pty=True)
    
    # Install pytorch3d
    conn.run("python3 -m pip install fvcore iopath", pty=True)l
    conn.run("python3 -m pip install pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py38_cu116_pyt1131/download.html", pty=True)

    # nstall Python Project Dependencies
    print("Installing remaining project dependencies...")
    setup_cmds = [
        # Fix PATH warning immediately
        "export PATH=$PATH:/home/cc/.local/bin",
        "python3 -m pip install cython ninja",
        "python3 -m pip install trimesh opencv-python tensorboard pandas tqdm matplotlib rich PyMCubes==0.1.4 numpy==1.24.4 open3d==0.19.0 gdown",
        "python3 -m pip install pysdf",
        "cd reproduce_miso && python3 -m pip install -e ."
    ]
    
    for cmd in setup_cmds:
        print(f"Running: {cmd}")
        conn.run(cmd, pty=True)

    # Final Verification
    print("Verifying installation...")
    conn.run("python3 -c 'import torch; import pytorch3d; import pysdf; print(\"Environment Fully Ready!\")'", pty=True)

print("\n SUCCESS! The environment is ready. You can now run your SLAM demo.")

Connecting to 129.114.109.70...
1. Terminating background update processes (fixing the lock error)...


  apt-daily.timer
  apt-daily-upgrade.timer


  3764

/var/lib/dpkg/lock-frontend:
dpkg: error: dpkg database lock is locked by another process


2. Cloning repository...


Cloning into 'reproduce_miso'...


3. Installing pip and system utilities...
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64  InRelease
Hit:2 http://security.ubuntu.com/ubuntu focal-security InRelease               
Hit:3 http://nova.clouds.archive.ubuntu.com/ubuntu focal InRelease             
Hit:4 http://nova.clouds.archive.ubuntu.com/ubuntu focal-updates InRelease
Hit:5 http://nova.clouds.archive.ubuntu.com/ubuntu focal-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree       
Reading state information... Done
python3-dev is already the newest version (3.8.2-0ubuntu2).
build-essential is already the newest version (12.8ubuntu1.1).
unzip is already the newest version (6.0-25ubuntu1.2).
python3-pip is already the newest version (20.0.2-5ubuntu1.11).
0 upgraded, 0 newly installed, 0 to remove and 93 not upgraded.
Upgrading pip...
Defaulting to user installation because normal site-packages is not writeable
4. Installing PyTorch, fvco

In [10]:
print("Downloading and extracting datasets...")

# Note: We use a multi-line string for the script
# We wrap it in a 'cd' command to ensure everything happens inside the repo
data_script = """
cd reproduce_miso
mkdir -p data/Newer_College results

# Newer College Dataset (Outdoor)
if [ ! -f "data/Newer_College/quad_e/poses_gt.txt" ]; then
    echo "Downloading Newer College Dataset..."
    python3 -m gdown --id 1_dPFKyaXASYaWMWeYWCCDrRdExFygj17 -O Newer_College.zip
    unzip -o -q Newer_College.zip -d data/Newer_College/
    # Fix nested folder structure
    [ -d "data/Newer_College/Newer_College" ] && mv data/Newer_College/Newer_College/* data/Newer_College/ && rmdir data/Newer_College/Newer_College
    rm Newer_College.zip
fi

# Pretrained Decoders
if [ ! -f "results/trained_decoders/decoder_quad.pt" ]; then
    echo "Downloading Pretrained Decoders..."
    python3 -m gdown --id 1d6R_DB14nEvhZ0rvtbzYCc4eQJAg49DT -O decoders.zip
    unzip -o -q decoders.zip -d results/ && rm decoders.zip
fi
"""

# Run the script using the 'conn' object
conn.run(data_script, pty=True)
print("Data Organized.")

/home/cc/.local/lib/python3.8/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1_dPFKyaXASYaWMWeYWCCDrRdExFygj17
From (redirected): https://drive.google.com/uc?id=1_dPFKyaXASYaWMWeYWCCDrRdExFygj17&confirm=t&uuid=d6851dea-a976-4c76-a1ee-ba243ca6a3fa
To: /home/cc/reproduce_miso/Newer_College.zip
100%|██████████████████████████████████████| 3.85G/3.85G [00:41<00:00, 93.8MB/s]
/home/cc/.local/lib/python3.8/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1d6R_DB14nEvhZ0rvtbzYCc4eQJAg49DT
To: /home/cc/reproduce_miso/decoders.zip
100%|██████████████████████████████████████| 38.1k/38.1k [00:00

In [11]:
print("Executing SLAM Pipeline (This may take several minutes)...")

with ssh.Remote(floating_ip) as conn:
    # We combine the cd and the run command to maintain the directory context
    conn.run("cd reproduce_miso && python3 demo/full_slam_newer_college.py", pty=True)

    print("Downloading result mesh...")
    # Get the file from remote path to local Jupyter path
    conn.get("reproduce_miso/results/demo/slam/ncd_quad/test/final_mesh.ply", "final_reconstruction.ply")

print("Processing complete. Visualizing, you can download the final_reconstruction.ply and visualize it outside")

Executing SLAM Pipeline (This may take several minutes)...
INFO:grid_opt.utils.utils:Create directory: ./results/demo/slam/ncd_quad/test
Read 1991 poses from file: data/Newer_College/quad_e/poses_gt.txt
Read 1991 poses from file: data/Newer_College/quad_e/poses_reg_icp.txt
INFO:grid_opt.datasets.sdf_3d_lidar:Read 1991 poses from files.
Dataset has 1991 usable frames.
Loading Lidar frames:   0%|                            | 0/1991 [00:00<?, ?it/s]/home/cc/reproduce_miso/grid_opt/utils/utils_geometry.py:325: UserWarning: scatter_reduce() is in beta and the API may change at any time. (Triggered internally at ../aten/src/ATen/native/TensorAdvancedIndexing.cpp:1615.)
  idx = torch.empty(
Sampling frames: 100%|█████████████████████| 1991/1991 [00:02<00:00, 774.86it/s]
INFO:grid_opt.datasets.sdf_3d_lidar:Constructed dataset with settings: 
near_surface_std=0.1, 
near_surface_n=0,
free_space_n=0, 
behind_surface_n=0, 
trunc_dist=0.5, 
distance_std=0.0, 
min_dist_ratio=0.5, 
voxel_size=0.600, 

# Visualize
To visualize `final_reconstruction.ply`, you can go to [here](https://imagetostl.com/view-ply-online) and just upload the downloaded .ply file to visualize it.

![Final reconstruction](ncquad.png)